# 01_eda.ipynb - EDA + Backtest Generalizado

**Pipeline**: Carga universo admitido → Features → Señales entrada → Backtest con Csl calibrado → Trades + Métricas

Configuración centralizada en `config/settings.py`

## 1. Imports y Configuración

In [4]:
import sys
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime

REPO_ROOT = Path.cwd().parent
sys.path.insert(0, str(REPO_ROOT))

# Configuración centralizada
from config.settings import DATA, FEATURES, RISK, BACKTEST, SIGNALS, get_universe_tickers

# Módulos core
from src.data import load_ohlc_from_yfinance, validate_ohlc_data
from src.features import add_all_features, calculate_efficiency_ratio, filter_momentum_entries
from src.risk import run_backtest_loop, calculate_performance_metrics

print(f"Config cargada: {DATA.source} {DATA.interval} {DATA.period}")
print(f"Capital: ${BACKTEST.capital:,.0f}, Max bars: {BACKTEST.max_bars}, TP/SL: {BACKTEST.tp_sl_ratio}:1")
print(f"Risk/trade: ${RISK.risk_per_trade:.0f}")

Config cargada: yfinance 1h 730d
Capital: $10,000, Max bars: 40, TP/SL: 1.5:1
Risk/trade: $100


## 2. Cargar Universo Admitido (Csl + BP calibrados)

In [5]:
admitted_path = REPO_ROOT / "data" / "universe_admitted.csv"
admitted_df = pd.read_csv(admitted_path)
print(f"Activos admitidos: {len(admitted_df)}")
admitted_df

Activos admitidos: 9


,ticker,csl,bp_median,bp_max,n_samples
0,MCRB,2.071067,1630.178739,2953.316136,500
1,LRMR,2.186679,1752.941510,2983.774098,500
2,ABSI,2.297152,1564.258837,2693.401827,500
3,IMUX,2.348268,1551.913088,2713.786004,500
4,HBIO,2.413816,1495.854857,2619.348096,500
5,HOTH,2.462243,1478.604231,2582.297380,500
6,KPTI,2.476287,1280.608848,2456.009761,500
7,SGMO,2.540506,915.892752,1721.758859,500
8,EBON,2.734792,1190.168119,1994.398966,500


## 3. Cargar Datos + Features + Señales + Backtest por Activo

In [6]:
all_trades = []
all_metrics = []

for _, row in admitted_df.iterrows():
    ticker = row['ticker']
    csl = row['csl']
    
    print(f"\n=== {ticker} (Csl={csl:.3f}) ===")
    
    try:
        # 1. Cargar datos OHLC
        df = load_ohlc_from_yfinance(ticker, period=DATA.period, interval=DATA.interval)
        
        if not validate_ohlc_data(df, min_rows=FEATURES.atr_period + RISK.lookforward_window + 100):
            print(f"  ❌ Datos insuficientes: {len(df)} barras")
            continue
        
        # 2. Generar todas las features (incluye ATR, ER, estructura, velas)
        df = add_all_features(
            df,
            atr_period=FEATURES.atr_period,
            sma_period=FEATURES.sma_period,
            roc_period=FEATURES.roc_period
        )
        
        # 3. Señal de entrada: 4 condiciones parametrizables
        signal = pd.Series(False, index=df.index)
        
        cond1 = pd.Series(True, index=df.index)
        if SIGNALS.use_er_filter:
            er = calculate_efficiency_ratio(df, k=FEATURES.er_k)
            cond1 = er > FEATURES.er_threshold
        
        cond2 = pd.Series(True, index=df.index)
        if SIGNALS.use_sma_trend:
            cond2 = df['Close'] > df['SMA']
        
        cond3 = pd.Series(True, index=df.index)
        if SIGNALS.use_structure_break:
            cond3 = df['Estructura_OK'] == True
        
        cond4 = pd.Series(True, index=df.index)
        if SIGNALS.use_volume_confirm:
            vol_ma = df['Volume'].rolling(FEATURES.vol_ma_period).mean()
            cond4 = df['Volume'] > FEATURES.vol_multiplier * vol_ma
        
        signal = cond1 & cond2 & cond3 & cond4
        
        n_signals = signal.sum()
        print(f"  Señales generadas: {n_signals}")
        
        if n_signals == 0:
            print(f"  ⚠️ Sin señales, saltando")
            continue
        
        # 4. Backtest con Csl calibrado del activo
        trades = run_backtest_loop(
            df=df,
            entry_signal=signal,
            c_sl=csl,
            c_tp=None,  # usa ratio 1.5 del config
            max_bars=BACKTEST.max_bars,
            risk_per_trade=RISK.risk_per_trade,
            capital=BACKTEST.capital
        )
        
        if len(trades) == 0:
            print(f"  ⚠️ Sin trades ejecutados")
            continue
        
        trades['ticker'] = ticker
        all_trades.append(trades)
        
        # 5. Métricas
        metrics = calculate_performance_metrics(trades, BACKTEST.capital)
        metrics['ticker'] = ticker
        metrics['csl'] = csl
        metrics['n_signals'] = n_signals
        metrics['n_trades'] = len(trades)
        all_metrics.append(metrics)
        
        print(f"  ✅ Trades: {len(trades)}, WinRate: {metrics.get('Win Rate', 0):.2%}, PF: {metrics.get('Profit Factor', 0):.2f}")
        
    except Exception as e:
        print(f"  ❌ Error {ticker}: {e}")
        continue


=== MCRB (Csl=2.071) ===
  Señales generadas: 257
  ✅ Trades: 122, WinRate: 35.25%, PF: 0.80

=== LRMR (Csl=2.187) ===
  Señales generadas: 288
  ✅ Trades: 130, WinRate: 40.00%, PF: 1.02

=== ABSI (Csl=2.297) ===
  Señales generadas: 289
  ✅ Trades: 140, WinRate: 40.00%, PF: 1.00

=== IMUX (Csl=2.348) ===
  Señales generadas: 267
  ✅ Trades: 114, WinRate: 42.11%, PF: 1.10

=== HBIO (Csl=2.414) ===
  Señales generadas: 215
  ✅ Trades: 106, WinRate: 35.85%, PF: 0.78

=== HOTH (Csl=2.462) ===
  Señales generadas: 171
  ✅ Trades: 87, WinRate: 33.33%, PF: 0.66

=== KPTI (Csl=2.476) ===
  Señales generadas: 248
  ✅ Trades: 105, WinRate: 36.19%, PF: 0.84

=== SGMO (Csl=2.541) ===
  Señales generadas: 233
  ✅ Trades: 115, WinRate: 37.39%, PF: 0.84

=== EBON (Csl=2.735) ===
  Señales generadas: 134
  ✅ Trades: 62, WinRate: 41.94%, PF: 1.07


## 4. Consolidar Resultados

In [7]:
if all_trades:
    trades_all = pd.concat(all_trades, ignore_index=True)
    metrics_all = pd.DataFrame(all_metrics)
    
    # Guardar
    out_dir = REPO_ROOT / "data"
    trades_all.to_csv(out_dir / "trades_all.csv", index=False)
    metrics_all.to_csv(out_dir / "metrics_all.csv", index=False)
    
    print(f"\n=== RESUMEN GLOBAL ===")
    print(f"Total trades: {len(trades_all)}")
    print(f"Activos con trades: {trades_all['ticker'].nunique()}")
    print(f"\nMétricas por activo:")
    print(metrics_all[['ticker', 'csl', 'n_trades', 'Win Rate', 'Profit Factor', 'Expectancy (USD)', 'Max Drawdown (%)', 'Sharpe Ratio', 'Retorno Total (%)']].to_string(index=False))
    
    # Métricas agregadas (portfolio simple equal weight)
    total_pnl = trades_all['PnL'].sum()
    total_ret = total_pnl / BACKTEST.capital * 100
    print(f"\n=== PORTFOLIO AGREGADO (equal weight) ===")
    print(f"PnL Total: ${total_pnl:,.2f}")
    print(f"Retorno Total: {total_ret:.2f}%")
    
    trades_all.head(10)
else:
    print("❌ No se generaron trades en ningún activo")


=== RESUMEN GLOBAL ===
Total trades: 981
Activos con trades: 9

Métricas por activo:
ticker      csl  n_trades  Win Rate  Profit Factor  Expectancy (USD)  Max Drawdown (%)  Sharpe Ratio  Retorno Total (%)
  MCRB 2.071067       122  0.352459       0.797717        -12.400453        -16.030784     -1.719146         -15.128553
  LRMR 2.186679       130  0.400000       1.024952          1.423002        -11.496055      0.188280           1.849903
  ABSI 2.297152       140  0.400000       0.997608         -0.137471        -12.811949     -0.018242          -0.192459
  IMUX 2.348268       114  0.421053       1.097898          5.554568        -11.427629      0.720856           6.332208
  HBIO 2.413816       106  0.358491       0.781015        -13.113094        -18.111189     -1.853073         -13.899880
  HOTH 2.462243        87  0.333333       0.658793        -21.262779        -23.156175     -3.117643         -18.498618
  KPTI 2.476287       105  0.361905       0.839584         -9.690337      

## 5. Análisis Básico de Calidad (Preview para 03_backtest_evaluation.ipynb)

In [8]:
if all_trades:
    # Distribución motivos de salida
    print("=== MOTIVOS DE SALIDA ===")
    print(trades_all['Motivo'].value_counts())
    
    # MAE / MFE promedio
    print(f"\n=== MAE/MFE PROMEDIO ===")
    print(f"MAE medio: ${trades_all['MAE'].mean():.2f}")
    print(f"MFE medio: ${trades_all['MFE'].mean():.2f}")
    print(f"Ratio MFE/MAE: {trades_all['MFE'].mean() / trades_all['MAE'].mean():.2f}")
    
    # PnL por motivo
    print(f"\n=== PnL POR MOTIVO ===")
    print(trades_all.groupby('Motivo')['PnL'].agg(['count', 'mean', 'sum']))
    
    # Duración media
    print(f"\nDuración media: {trades_all['Velas'].mean():.1f} barras")
    print(f"Max duración: {trades_all['Velas'].max()} barras")

=== MOTIVOS DE SALIDA ===
Motivo
SL           562
TP           338
Time Exit     81
Name: count, dtype: int64

=== MAE/MFE PROMEDIO ===
MAE medio: $0.52
MFE medio: $0.56
Ratio MFE/MAE: 1.07

=== PnL POR MOTIVO ===
           count        mean           sum
Motivo                                    
SL           562  -99.706404 -56034.999298
TP           338  149.626565  50573.778904
Time Exit     81   -3.971771   -321.713461

Duración media: 12.7 barras
Max duración: 40 barras
